In [ ]:
import os
import numpy as np
import random
random.seed = 42
from path import Path
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier  
import ot

In [ ]:
def read_off(file):
    if 'OFF' != file.readline().strip():
        raise('Not a valid OFF header')
    n_verts, n_faces, __ = tuple([int(s) for s in file.readline().strip().split(' ')])
    verts = [[float(s) for s in file.readline().strip().split(' ')] for i_vert in range(n_verts)]
    faces = [[int(s) for s in file.readline().strip().split(' ')][1:] for i_face in range(n_faces)]
    return verts, faces


def visualize_rotate(data):
    x_eye, y_eye, z_eye = 1.25, 1.25, 0.8
    frames=[]

    def rotate_z(x, y, z, theta):
        w = x+1j*y
        return np.real(np.exp(1j*theta)*w), np.imag(np.exp(1j*theta)*w), z

    for t in np.arange(0, 10.26, 0.1):
        xe, ye, ze = rotate_z(x_eye, y_eye, z_eye, -t)
        frames.append(dict(layout=dict(scene=dict(camera=dict(eye=dict(x=xe, y=ye, z=ze))))))
    fig = go.Figure(data=data,
        layout=go.Layout(
            updatemenus=[dict(type='buttons',
                showactive=False,
                y=1,
                x=0.8,
                xanchor='left',
                yanchor='bottom',
                pad=dict(t=45, r=10),
                buttons=[dict(label='Play',
                    method='animate',
                    args=[None, dict(frame=dict(duration=50, redraw=True),
                        transition=dict(duration=0),
                        fromcurrent=True,
                        mode='immediate'
                        )]
                    )
                ])]
        ),
        frames=frames
    )

    return fig


def pcshow(xs,ys,zs):
    data=[go.Scatter3d(x=xs, y=ys, z=zs,
                                   mode='markers')]
    fig = visualize_rotate(data)
    fig.update_traces(marker=dict(size=2,
                      line=dict(width=2,
                      color='DarkSlateGrey')),
                      selector=dict(mode='markers'))
    fig.show()

In [ ]:
class PointSampler(object):
    def __init__(self, output_size):
        assert isinstance(output_size, int)
        self.output_size = output_size
    
    def triangle_area(self, pt1, pt2, pt3):
        side_a = np.linalg.norm(pt1 - pt2)
        side_b = np.linalg.norm(pt2 - pt3)
        side_c = np.linalg.norm(pt3 - pt1)
        s = 0.5 * ( side_a + side_b + side_c)
        return max(s * (s - side_a) * (s - side_b) * (s - side_c), 0)**0.5

    def sample_point(self, pt1, pt2, pt3):

        s, t = sorted([random.random(), random.random()])
        f = lambda i: s * pt1[i] + (t-s)*pt2[i] + (1-t)*pt3[i]
        return (f(0), f(1), f(2))
        
    
    def __call__(self, mesh):
        verts, faces = mesh
        verts = np.array(verts)
        areas = np.zeros((len(faces)))

        for i in range(len(areas)):
            areas[i] = (self.triangle_area(verts[faces[i][0]],
                                           verts[faces[i][1]],
                                           verts[faces[i][2]]))
            
        sampled_faces = (random.choices(faces, 
                                      weights=areas,
                                      cum_weights=None,
                                      k=self.output_size))
        
        sampled_points = np.zeros((self.output_size, 3))

        for i in range(len(sampled_faces)):
            sampled_points[i] = (self.sample_point(verts[sampled_faces[i][0]],
                                                   verts[sampled_faces[i][1]],
                                                   verts[sampled_faces[i][2]]))
        
        return sampled_points


class Normalize(object):
    def __call__(self, pointcloud):
        assert len(pointcloud.shape)==2
        
        norm_pointcloud = pointcloud - np.mean(pointcloud, axis=0) 
        norm_pointcloud /= np.max(np.linalg.norm(norm_pointcloud, axis=1))

        return  norm_pointcloud

In [ ]:
path = Path("D:/PYTHON_code/HCP-main/3D_point_classification/data/ModelNet10")

folders = [dir for dir in sorted(os.listdir(path)) if os.path.isdir(path/dir)]
classes = {folder: i for i, folder in enumerate(folders)};
 
print(classes)

data = pd.read_csv('metadata_modelnet10.csv')
data.columns = ['object_id', 'type', 'split', 'object_path']
data.head(5)

In [ ]:
print(np.sum(data.split=='train'))
train_data = data[data['split']=='train'].reset_index(drop=True)
print(np.sum(data.split=='test'))
test_data = data[data['split']=='test'].reset_index(drop=True)

In [ ]:
seed = 6
np.random.seed(seed)
subclass = classes.keys()
print(subclass)
Ktrain = 1*10

train_3D = [None]*Ktrain
train_label = np.zeros(Ktrain)
i = 0
j = 0

KK = 3000

for category in subclass:


    temple_data = train_data[train_data['type']==category].sample(n=int(Ktrain/10),random_state=seed).reset_index(drop=True)

    for k in range(temple_data.shape[0]):
        temple_path = temple_data.object_path[k]
        with open(path/temple_path, 'r') as f:
            verts, faces = read_off(f)
            pointcloud = PointSampler(KK)((verts, faces))
            train_3D[i] = Normalize()(pointcloud)
            train_label[i] = j
            i += 1
    
    print(k+1)
    j += 1
    print(i)


In [ ]:
import numpy as np

def random_transformation_matrix(seed):
    np.random.seed(seed)
    # 随机生成绕x、y、z轴的旋转角（单位：弧度）
    angle_x = np.random.uniform(0, 2 * np.pi)
    angle_y = np.random.uniform(0, 2 * np.pi)
    angle_z = np.random.uniform(0, 2 * np.pi)
    
    # 绕x轴的旋转矩阵
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(angle_x), -np.sin(angle_x)],
        [0, np.sin(angle_x), np.cos(angle_x)]
    ])
    
    # 绕y轴的旋转矩阵
    Ry = np.array([
        [np.cos(angle_y), 0, np.sin(angle_y)],
        [0, 1, 0],
        [-np.sin(angle_y), 0, np.cos(angle_y)]
    ])
    
    # 绕z轴的旋转矩阵
    Rz = np.array([
        [np.cos(angle_z), -np.sin(angle_z), 0],
        [np.sin(angle_z), np.cos(angle_z), 0],
        [0, 0, 1]
    ])
    
    # 组合旋转矩阵（按照 Rz * Ry * Rx 的顺序组合）
    R = Rz @ Ry @ Rx

    # 随机生成平移向量，这里平移范围设定为[-1, 1]
    #t = np.random.uniform(-1, 1, size=(3, 1))
    t = np.zeros((3,1))
    
    # 构造4x4变换矩阵，最后一行固定为 [0, 0, 0, 1]
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3:] = t
    
    return T


In [ ]:
def transform_point_cloud(points, transformation_matrix):

    if points.shape[1] != 3:
        raise ValueError("点云的形状必须为 (n, 3)")
    if transformation_matrix.shape != (4, 4):
        raise ValueError("变换矩阵的形状必须为 (4, 4)")

    # 将点云扩展为齐次坐标 (n, 4)，最后一列为 1
    n = points.shape[0]
    points_homogeneous = np.hstack((points, np.ones((n, 1))))  # (n, 4)
    transformed_points_homogeneous = np.dot(transformation_matrix, points_homogeneous.T).T  # (n, 4)
    transformed_points = transformed_points_homogeneous[:, :3]
    return transformed_points

In [ ]:
t = 15
s = 5
rep = t + s
train_3D_small = []
train_label_small = []
test_3D_small = []
test_label_small = []


for j in range(len(train_3D)):
    for i in range(t):
        train_3D_small.append(transform_point_cloud(train_3D[j],random_transformation_matrix(i+j)))
        train_label_small = np.append(train_label_small,train_label[j])
    for q in range(t,rep,1):
        test_3D_small.append(transform_point_cloud(train_3D[j],random_transformation_matrix(q+j)))
        test_label_small = np.append(test_label_small,train_label[j])


# 抽样
p = 3000
noise_scale = 0.2*train_3D[1].mean()
for k in range(len(train_3D_small)):
    np.random.seed(1)
    indices = np.random.choice(len(train_3D_small[k]), size=p, replace=False)
    train_3D_small[k] = train_3D_small[k][indices]
    np.random.seed(k)
    noise = noise_scale * np.random.randn(len(train_3D_small[k]), 3)
    train_3D_small[k] = train_3D_small[k] + noise

for k in range(len(test_3D_small)):
    np.random.seed(1)
    indices = np.random.choice(len(test_3D_small[k]), size=p, replace=False)
    test_3D_small[k] = test_3D_small[k][indices]
    np.random.seed(k+len(test_3D_small))
    noise = noise_scale * np.random.randn(len(test_3D_small[k]), 3)
    test_3D_small[k] = test_3D_small[k] + noise

Ktrain = len(train_3D_small)
Ktest = len(test_3D_small)

In [ ]:
import numpy as np
import ot
import ot.plot
import scipy as sp
def generate_RD(num_points, num_clusters):
    cluster_centers = np.random.rand(num_clusters)
    std_dev = np.random.uniform(0, 0.05, num_clusters)
    cluster_sizes = np.random.multinomial(num_points, np.ones(num_clusters) / num_clusters)

    points = []
    for i in range(num_clusters):
        cluster_points = cluster_centers[i] + np.random.randn(cluster_sizes[i]) * std_dev[i]
        points.append(cluster_points)
    return np.concatenate(points)

def sort_and_match_single_index(vec1, vec2):
    # 检查长度是否相等
    if len(vec1) != len(vec2):
        raise ValueError("两个向量的长度必须相等")
    
    # 对 vec1 排序，获取排序索引
    sorted_indices_vec1 = np.argsort(vec1)
    sorted_indices_vec1 = np.argsort(sorted_indices_vec1)
    sorted_indices_vec2 = np.argsort(vec2)
    
    return sorted_indices_vec1, sorted_indices_vec2

def get_length5(plength, npoints):
    indices1, indices2 = sort_and_match_single_index(plength, npoints)
    length = np.abs(plength - npoints[indices2][indices1])
    return length

def row_softmax(matrix):
    row_max = np.max(matrix, axis=1, keepdims=True)
    exp_matrix = np.exp(matrix - row_max)
    #exp_matrix = matrix
    softmax_matrix = exp_matrix / np.sum(exp_matrix, axis=1, keepdims=True)
    return softmax_matrix


def RIOT(
        X,
        Y,
        lp = 2,
        rep = 50,
        method = 'emd',
        determin = True,
        rd_rad = None,
        rd_peak = 3,
        softmax = False,
        maxed = False,
        sliced = True,
        reg = 0.01,
        threepeaks = True
):
    """
    :param X: 初始点云 n*p
    :param Y: 初始点云 m*q
    :param lp: distance 范数
    :param rep: reference distribution 数量
    :param method: 最后coupling的计算方法
    :param determin: 是否选用确定的 reference distance
    :param rd_rad: reference distribution的范围
    :param rd_peak: reference distribution的峰数
    :param softmax: 是否进行softmax
    :param maxed: 是否选择取max
    :param reg:(如果为sinkhorn)对应的reg设定
    :return: RIOT的loss以及对应coupling
    """

    # Initialize
    n, p = np.shape(X)
    m, q = np.shape(Y) 
    Xn = (X - X.mean(axis=0)) / (X - X.mean(axis=0)).std()
    Yn = (Y - Y.mean(axis=0)) / (Y - Y.mean(axis=0)).std()

    # OT length
    X_plength = np.linalg.norm(Xn, axis = 1)
    Y_plength = np.linalg.norm(Yn, axis = 1)
    vectors1 = []
    vectors2 = []
    if rd_rad == None:
        rd_rad = (X_plength.mean() + Y_plength.mean() + X_plength.std() + X_plength.std())/2
        print(rd_rad)
    for k in range(rep):
        if (determin):
            np.random.seed(k)
        if (threepeaks):
            npoints = rd_rad * generate_RD(n, rd_peak)
            
        else:
            npoints = rd_rad * np.random.rand(n)
            
        length1 = get_length5(X_plength, npoints)
        length2 = get_length5(Y_plength, npoints)
        vectors1.append(length1)
        vectors2.append(length2)
    X1 = np.vstack(vectors1).T
    Y1 = np.vstack(vectors2).T

    # Distance
    X_dist = sp.spatial.distance.cdist(Xn,Xn)
    Y_dist = sp.spatial.distance.cdist(Yn,Yn)
    X_dist = X_dist / X_dist.std()
    Y_dist = Y_dist / Y_dist.std()

    # ttfeature
    if(softmax):
        if(sliced):
            X2 = row_softmax(X_dist) @ X1
            Y2 = row_softmax(Y_dist) @ Y1
        else:
            X2 = row_softmax(X_dist) @ X1
            Y2 = row_softmax(Y_dist) @ Y1   
    else:
        if(sliced):
            X2 = X_dist @ X1
            Y2 = Y_dist @ Y1
        else:
            X2 = X_dist @ X1
            Y2 = Y_dist @ Y1


    # OT
    if(sliced):
        X2_indices = np.argsort(X2, axis=0)
        Y2_indices = np.argsort(Y2, axis=0)   
        X2 = np.sort(X2, axis=0)/X1.mean(axis=0)
        Y2 = np.sort(Y2, axis=0)/Y1.mean(axis=0)
        col_losses = np.sum((X2 - Y2) ** 2, axis=0)
        #loss
        if(maxed):
            max_col_idx = np.argmax(col_losses)
            loss = np.sqrt(col_losses[max_col_idx]/((n+m)/2)**2)
            X2_orig_indices = X2_indices[:, max_col_idx]
            Y2_orig_indices = Y2_indices[:, max_col_idx]
            P = np.zeros((n,m), dtype=float)
            P[X2_orig_indices, Y2_orig_indices] = 1/n
        else:
            loss = np.sqrt(np.mean(col_losses)/((n+m)/2)**2)
            P = 'Sliced version without coupling'
    else:
        if((method == 'sinkhorn') | (method == 'emd')):
            a = np.ones(n) / n
            b = np.ones(m) / m
            C = ot.dist(X2, Y2, 'euclidean')
            C = C / C.mean()
        if(method == 'sinkhorn'):
            P = ot.sinkhorn(a, b, C, reg=reg, stopThr=1e-9)
        elif(method == 'emd'):
            P = ot.emd(a, b, C)
        # loss
        loss = (C*P).sum()
    
    return P, loss


In [ ]:
import numpy as np

def calculate_distance_matrix(points: np.ndarray) -> np.ndarray:
    diff = points[:, None, :] - points[None, :, :]
    distance_matrix = np.linalg.norm(diff, axis=-1)
    return distance_matrix

def GW_distance(D1: np.ndarray,
                D2: np.ndarray,
                p: np.ndarray,
                q: np.ndarray,
                P: np.ndarray) -> float:

    D1 = np.asarray(D1, dtype=float)
    D2 = np.asarray(D2, dtype=float)
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    P = np.asarray(P, dtype=float)

    n = p.shape[0]
    m = q.shape[0]

    const_1 = (D1**2) @ p.reshape(-1, 1) @ np.ones((1, m))
    const_2 = np.ones((n, 1)) @ (q.reshape(1, -1) @ (D2**2).T)
    const = const_1 + const_2

    L = const - 2 * (D1 @ P @ D2)

    loss = np.sum(L * P)
    return loss

def gromov_wasserstein_loss(
    X1: np.ndarray,
    X2: np.ndarray,
    p: np.ndarray,
    q: np.ndarray,
    P: np.ndarray
) -> float:

    dm1 = calculate_distance_matrix(X1)
    dm2 = calculate_distance_matrix(X2)
    dm1 = dm1/dm1.max()
    dm2 = dm2/dm2.max()

    gw_loss = GW_distance(dm1, dm2, p, q, P)
    return gw_loss



In [ ]:
i = 15
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(train_3D_small[i][:,0],train_3D_small[i][:,1],train_3D_small[i][:,2])

In [ ]:
import time
import numpy as np
from joblib import Parallel, delayed
from sklearn.neighbors import KNeighborsClassifier

t = time.time()

# ✅ 并行计算 rot_dist_matrix（Ktrain × Ktrain）
def compute_train_pair(i, j):
    _, dist = RIOT(train_3D_small[i], train_3D_small[j], rd_rad=2, sliced=True,
                   softmax=False, rep=50, maxed=False, threepeaks=True)
    return (i, j, dist)

train_index_pairs = [(i, j) for i in range(Ktrain) for j in range(i + 1, Ktrain)]
results_train = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_train_pair)(i, j) for i, j in train_index_pairs
)

rot_dist_matrix = np.zeros((Ktrain, Ktrain))
for i, j, dist in results_train:
    rot_dist_matrix[i, j] = dist
    rot_dist_matrix[j, i] = dist  # 对称填充

# ✅ 并行计算 rot_dist_matrix2（Ktest × Ktrain）
def compute_test_pair(i, j):
    _, dist = RIOT(test_3D_small[i], train_3D_small[j], rd_rad=2, sliced=True,
                   softmax=False, rep=50, maxed=False, threepeaks=True)
    return (i, j, dist)

test_index_pairs = [(i, j) for i in range(Ktest) for j in range(Ktrain)]
results_test = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_test_pair)(i, j) for i, j in test_index_pairs
)

rot_dist_matrix2 = np.zeros((Ktest, Ktrain))
for i, j, dist in results_test:
    rot_dist_matrix2[i, j] = dist

# ✅ KNN 分类与评估
estimator = KNeighborsClassifier(metric='precomputed', n_neighbors=5)
estimator.fit(rot_dist_matrix, train_label_small)

print('Test Accuracy is ',
      np.sum(estimator.predict(rot_dist_matrix2) == test_label_small) / Ktest * 100, '%')
print('Train Accuracy is ',
      np.sum(estimator.predict(rot_dist_matrix) == train_label_small) / Ktrain * 100, '%')

print('Time is ', time.time() - t, 'seconds')


In [ ]:
import time
import numpy as np
from joblib import Parallel, delayed
from sklearn.neighbors import KNeighborsClassifier

t = time.time()

# ✅ 并行计算 rot_dist_matrix（Ktrain × Ktrain）
def compute_train_pair(i, j):
    _, dist = RIOT(train_3D_small[i], train_3D_small[j], rd_rad=2, sliced=True,
                   softmax=False, rep=50, maxed=True, threepeaks=True)
    return (i, j, dist)

train_index_pairs = [(i, j) for i in range(Ktrain) for j in range(i + 1, Ktrain)]
results_train = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_train_pair)(i, j) for i, j in train_index_pairs
)

rot_dist_matrix = np.zeros((Ktrain, Ktrain))
for i, j, dist in results_train:
    rot_dist_matrix[i, j] = dist
    rot_dist_matrix[j, i] = dist  # 对称填充

# ✅ 并行计算 rot_dist_matrix2（Ktest × Ktrain）
def compute_test_pair(i, j):
    _, dist = RIOT(test_3D_small[i], train_3D_small[j], rd_rad=2, sliced=True,
                   softmax=False, rep=50, maxed=True, threepeaks=True)
    return (i, j, dist)

test_index_pairs = [(i, j) for i in range(Ktest) for j in range(Ktrain)]
results_test = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_test_pair)(i, j) for i, j in test_index_pairs
)

rot_dist_matrix2 = np.zeros((Ktest, Ktrain))
for i, j, dist in results_test:
    rot_dist_matrix2[i, j] = dist

# ✅ KNN 分类与评估
estimator = KNeighborsClassifier(metric='precomputed', n_neighbors=5)
estimator.fit(rot_dist_matrix, train_label_small)

print('Test Accuracy is ',
      np.sum(estimator.predict(rot_dist_matrix2) == test_label_small) / Ktest * 100, '%')
print('Train Accuracy is ',
      np.sum(estimator.predict(rot_dist_matrix) == train_label_small) / Ktrain * 100, '%')

print('Time is ', time.time() - t, 'seconds')



In [ ]:
import time
import numpy as np
import scipy as sp
import ot
from sklearn.neighbors import KNeighborsClassifier
from joblib import Parallel, delayed

t = time.time()

# ✅ 训练集对之间的EGW距离（对称矩阵）
def compute_egw_train(i, j):
    D11 = sp.spatial.distance.cdist(train_3D_small[i], train_3D_small[i])
    D12 = sp.spatial.distance.cdist(train_3D_small[j], train_3D_small[j])
    D11 /= D11.max()
    D12 /= D12.max()
    p = ot.unif(len(train_3D_small[i]))
    q = ot.unif(len(train_3D_small[j]))
    dist = ot.gromov.entropic_gromov_wasserstein(
        D11, D12, p, q, loss_fun="square_loss", epsilon=0.01, log=True, verbose=False
    )[1]["gw_dist"]
    return (i, j, dist)

train_pairs = [(i, j) for i in range(Ktrain) for j in range(i + 1, Ktrain)]
results_train = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_egw_train)(i, j) for i, j in train_pairs
)

egw_dist_matrix = np.zeros((Ktrain, Ktrain))
for i, j, dist in results_train:
    egw_dist_matrix[i, j] = dist
    egw_dist_matrix[j, i] = dist  # 对称性


# ✅ 测试集与训练集之间的EGW距离
def compute_egw_test(i, j):
    D11 = sp.spatial.distance.cdist(test_3D_small[i], test_3D_small[i])
    D12 = sp.spatial.distance.cdist(train_3D_small[j], train_3D_small[j])
    D11 /= D11.max()
    D12 /= D12.max()
    p = ot.unif(len(test_3D_small[i]))
    q = ot.unif(len(train_3D_small[j]))
    dist = ot.gromov.entropic_gromov_wasserstein(
        D11, D12, p, q, loss_fun="square_loss", epsilon=0.01, log=True, verbose=False
    )[1]["gw_dist"]
    return (i, j, dist)

test_pairs = [(i, j) for i in range(Ktest) for j in range(Ktrain)]
results_test = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_egw_test)(i, j) for i, j in test_pairs
)

egw_dist_matrix2 = np.zeros((Ktest, Ktrain))
for i, j, dist in results_test:
    egw_dist_matrix2[i, j] = dist

# ✅ KNN 预测与评估
estimator = KNeighborsClassifier(metric='precomputed', n_neighbors=5)
estimator.fit(egw_dist_matrix, train_label_small)

print('Test Accuracy is ',
      np.sum(estimator.predict(egw_dist_matrix2) == test_label_small) / Ktest * 100, '%')
print('Train Accuracy is ',
      np.sum(estimator.predict(egw_dist_matrix) == train_label_small) / Ktrain * 100, '%')

print('Time is ', time.time() - t, 'seconds')


In [ ]:
import time
import numpy as np
import torch
from sklearn.neighbors import KNeighborsClassifier
import sys
sys.path.append("D:\PYTHON_code\SGW-master\lib")
from risgw import risgw_gpu
import torch
from joblib import Parallel, delayed

t = time.time()

np.random.seed(seed)

# ✅ 并行计算训练集的 SGW 距离矩阵
def compute_sgw_train(i, j):
    xi = torch.from_numpy(train_3D_small[i]).to(torch.float32)
    xj = torch.from_numpy(train_3D_small[j]).to(torch.float32)
    dist = risgw_gpu(xi, xj, device='cpu', proj_num=50)
    return (i, j, dist)

train_pairs = [(i, j) for i in range(Ktrain) for j in range(i + 1, Ktrain)]
results_train = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_sgw_train)(i, j) for i, j in train_pairs
)

sgw_dist_matrix = np.zeros((Ktrain, Ktrain))
for i, j, dist in results_train:
    sgw_dist_matrix[i, j] = dist
    sgw_dist_matrix[j, i] = dist  # 对称填充


# ✅ 并行计算测试集与训练集之间的 SGW 距离矩阵
def compute_sgw_test(i, j):
    xi = torch.from_numpy(test_3D_small[i]).to(torch.float32)
    xj = torch.from_numpy(train_3D_small[j]).to(torch.float32)
    dist = risgw_gpu(xi, xj, device='cpu', proj_num=50)
    return (i, j, dist)

test_pairs = [(i, j) for i in range(Ktest) for j in range(Ktrain)]
results_test = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_sgw_test)(i, j) for i, j in test_pairs
)

sgw_dist_matrix2 = np.zeros((Ktest, Ktrain))
for i, j, dist in results_test:
    sgw_dist_matrix2[i, j] = dist

# ✅ KNN 分类与评估
estimator = KNeighborsClassifier(metric='precomputed', n_neighbors=5)
estimator.fit(sgw_dist_matrix, train_label_small)

print('Test Accuracy is ',
      np.sum(estimator.predict(sgw_dist_matrix2) == test_label_small) / Ktest * 100, '%')
print('Train Accuracy is ',
      np.sum(estimator.predict(sgw_dist_matrix) == train_label_small) / Ktrain * 100, '%')

print('Time is ', time.time() - t, 'seconds')


In [ ]:
import time
import numpy as np
import scipy as sp
import ot
from sklearn.neighbors import KNeighborsClassifier
from joblib import Parallel, delayed

t = time.time()
np.random.seed(seed)

# ✅ 并行计算训练集 GW 距离
def compute_gw_train(i, j):
    C3 = sp.spatial.distance.cdist(train_3D_small[i], train_3D_small[i])
    C4 = sp.spatial.distance.cdist(train_3D_small[j], train_3D_small[j])
    C3 /= C3.max()
    C4 /= C4.max()
    p = ot.unif(len(train_3D_small[i]))
    q = ot.unif(len(train_3D_small[j]))
    dist = ot.gromov.gromov_wasserstein(
        C3, C4, p, q, "square_loss", verbose=False, log=True
    )[1]["gw_dist"]
    return (i, j, dist)

train_pairs = [(i, j) for i in range(Ktrain) for j in range(i + 1, Ktrain)]
results_train = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_gw_train)(i, j) for i, j in train_pairs
)

gw_dist_matrix = np.zeros((Ktrain, Ktrain))
for i, j, dist in results_train:
    gw_dist_matrix[i, j] = dist
    gw_dist_matrix[j, i] = dist  # 对称性


# ✅ 并行计算测试集对训练集的 GW 距离
def compute_gw_test(i, j):
    C1 = sp.spatial.distance.cdist(test_3D_small[i], test_3D_small[i])
    C2 = sp.spatial.distance.cdist(train_3D_small[j], train_3D_small[j])
    C1 /= C1.max()
    C2 /= C2.max()
    p = ot.unif(len(test_3D_small[i]))
    q = ot.unif(len(train_3D_small[j]))
    dist = ot.gromov.gromov_wasserstein(
        C1, C2, p, q, "square_loss", verbose=False, log=True
    )[1]["gw_dist"]
    return (i, j, dist)

test_pairs = [(i, j) for i in range(Ktest) for j in range(Ktrain)]
results_test = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_gw_test)(i, j) for i, j in test_pairs
)

gw_dist_matrix2 = np.zeros((Ktest, Ktrain))
for i, j, dist in results_test:
    gw_dist_matrix2[i, j] = dist

# ✅ 修正数值问题：去掉负数
gw_dist_matrix[gw_dist_matrix < 0] = 0
gw_dist_matrix2[gw_dist_matrix2 < 0] = 0

# ✅ KNN 分类与评估
estimator = KNeighborsClassifier(metric='precomputed', n_neighbors=5)
estimator.fit(gw_dist_matrix, train_label_small)

print('Test Accuracy is ',
      np.sum(estimator.predict(gw_dist_matrix2) == test_label_small) / Ktest * 100, '%')
print('Train Accuracy is ',
      np.sum(estimator.predict(gw_dist_matrix) == train_label_small) / Ktrain * 100, '%')

print('Time is ', time.time() - t, 'seconds')
